# Example Animal: Empirical vs Model Fit

Pick one well-assigned animal and show what the fit looks like:
- Pooled update matrix: Real | BE | SC
- Psychometric overlay: Real vs best-fit model

This is the "what a good fit looks like" slide — goes BEFORE
the aggregate validation and model assignment results.

**Depends on:** Real GS results (NB 20) or SBI results (NB 21),
plus real animal data.

In [ ]:
%matplotlib inline
from shared_setup import *
import pickle
from pathlib import Path

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
CV_DIR = Path('../results/cv')
SBI_DIR = Path('../results/sbi_static')
CONFIG_PATH = Path('../config.yaml')  # adjust to your local config

# Pick an example animal — ideally one where all methods agree
# and the margin is clear. Update after running NB 20/22.
EXAMPLE_ANIMAL = 'SS05'  # ← change based on your results
FIT_TARGET = 'update_matrix'
DISTRIBUTION = 'uniform'
STAGE = 'Full_Task_Cont'

## 1. Load Best-Fit Parameters

Try SBI params first (posterior median), fall back to GS.

In [ ]:
be_params = None
sc_params = None
param_source = None

# ── Try SBI posterior params ───────────────────────────────────────────────
sbi_posterior_path = SBI_DIR / DISTRIBUTION / f'animal_{EXAMPLE_ANIMAL}.pkl'
if sbi_posterior_path.exists():
    with open(sbi_posterior_path, 'rb') as f:
        sbi_data = pickle.load(f)
    be_params = sbi_data['be_params']
    sc_params = sbi_data['sc_params']
    param_source = 'SBI posterior median'
    print(f'Loaded SBI params for {EXAMPLE_ANIMAL}')

# ── Fall back to GS best params ───────────────────────────────────────────
if be_params is None:
    for model in ['BE', 'SC']:
        gs_path = CV_DIR / f'{DISTRIBUTION}_{FIT_TARGET}' / f'cv_{EXAMPLE_ANIMAL}_{model}.pkl'
        if gs_path.exists():
            with open(gs_path, 'rb') as f:
                gs_data = pickle.load(f)
            # Get best seed's params
            results = gs_data.get('results', [])
            valid = [r for r in results
                     if not np.isnan(r.get('avg_test_error', np.nan))
                     and 'best_params_single' in r and r['best_params_single']]
            if valid:
                best = min(valid, key=lambda r: r['avg_test_error'])
                if model == 'BE':
                    be_params = best['best_params_single']
                else:
                    sc_params = best['best_params_single']
    if be_params is not None:
        param_source = 'GS best seed'
        print(f'Loaded GS params for {EXAMPLE_ANIMAL}')

if be_params is None or sc_params is None:
    print('WARNING: Could not load params. Check paths.')
else:
    print(f'Source: {param_source}')
    print(f'BE: {be_params}')
    print(f'SC: {sc_params}')

## 2. Load Animal Data and Build FittingData

In [ ]:
from scripts.config import load_animal_data
from behav_utils.data.selection import select_sessions, fitting_data_from_sessions

animal = load_animal_data(EXAMPLE_ANIMAL, config_path=CONFIG_PATH)
sessions = select_sessions(animal, 'expert_uniform')
print(f'{EXAMPLE_ANIMAL}: {len(sessions)} expert uniform sessions')

fd = fitting_data_from_sessions(sessions, EXAMPLE_ANIMAL)
print(f'FittingData: {fd.n_sessions} sessions, '
      f'{fd.trials_per_session.sum()} total trials')

## 3. Simulate Both Models on Real Stimuli

In [ ]:
from inference.comparison import simulate_all_sessions

print('Simulating (this takes a minute)...')
session_data = simulate_all_sessions(
    fd, be_params, sc_params,
    burn_in=1000, n_reps=20, n_bins=8, seed=42,
)
print(f'Simulated {len(session_data)} sessions')

## 4. Pooled UM Comparison (Slide 27)

In [ ]:
from plotting.comparison import plot_pooled_um_comparison

fig = plot_pooled_um_comparison(session_data, EXAMPLE_ANIMAL)
if fig is not None:
    plt.show()

## 5. Psychometric Overlay

In [ ]:
from behav_utils.analysis.psychometry import fit_psychometric
from behav_utils.analysis.utils import cumulative_gaussian

# Pool all sessions
all_stim = np.concatenate([d['stimuli'] for d in session_data])
all_cat = np.concatenate([d['categories'] for d in session_data])
all_ch = np.concatenate([d['choices'] for d in session_data])

x_fine = np.linspace(-1.1, 1.1, 200)

fig, ax = plt.subplots(figsize=(7, 5))

# Binned real data
n_bins = 8
bin_edges = np.linspace(-1, 1, n_bins + 1)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_idx = np.digitize(all_stim, bin_edges) - 1
bin_idx = np.clip(bin_idx, 0, n_bins - 1)
bin_means = np.array([
    np.nanmean(all_ch[bin_idx == b]) for b in range(n_bins)
])
bin_counts = np.array([
    np.sum(bin_idx == b) for b in range(n_bins)
])
bin_sem = np.array([
    np.nanstd(all_ch[bin_idx == b]) / np.sqrt(max(1, np.sum(bin_idx == b)))
    for b in range(n_bins)
])

ax.errorbar(bin_centres, bin_means, yerr=bin_sem,
            fmt='ko', markersize=6, capsize=3, label='Data', zorder=10)

# Fit and plot psychometric curves
colours = {'Real': 'black', 'BE': '#4C72B0', 'SC': '#DD8452'}
lws = {'Real': 2.5, 'BE': 2.0, 'SC': 2.0}

# Real fit
real_fit = fit_psychometric(all_stim, all_ch)
if real_fit.get('success', False):
    y = cumulative_gaussian(
        x_fine, real_fit['mu'], real_fit['sigma'],
        real_fit['lapse_low'], real_fit['lapse_high'],
    )
    ax.plot(x_fine, y, '-', color=colours['Real'], lw=lws['Real'],
            label=f'Real (PSE={real_fit["mu"]:.3f})')

# Model fits: simulate pooled data from each model
from models.BE_core import BEParams, BEModel
from models.SC_core import SCParams, SCModel

for model_name, params_dict, Model, Params in [
    ('BE', be_params, BEModel, BEParams),
    ('SC', sc_params, SCModel, SCParams),
]:
    # Average psychometric across sessions
    all_model_ch = []
    for d in session_data:
        key = f'{model_name.lower()}_um'  # we just need the choices
        # Use the be_psych / sc_psych from session_data
        pass

    # Use the pre-computed session-level psychometric fits
    psych_key = f'{model_name.lower()}_psych'
    psychs = [d[psych_key] for d in session_data
              if d[psych_key].get('success', False)]
    if psychs:
        mean_mu = np.mean([p['mu'] for p in psychs])
        mean_sigma = np.mean([p['sigma'] for p in psychs])
        mean_ll = np.mean([p['lapse_low'] for p in psychs])
        mean_lh = np.mean([p['lapse_high'] for p in psychs])
        y = cumulative_gaussian(x_fine, mean_mu, mean_sigma, mean_ll, mean_lh)
        ax.plot(x_fine, y, '-', color=colours[model_name], lw=lws[model_name],
                label=f'{model_name} (PSE={mean_mu:.3f})')

ax.axhline(0.5, color='grey', ls='--', alpha=0.3)
ax.axvline(0, color='grey', ls='--', alpha=0.3)
ax.set_xlabel('Stimulus', fontsize=11)
ax.set_ylabel('P(choose B)', fontsize=11)
ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.set_title(f'{EXAMPLE_ANIMAL} — Psychometric Comparison ({len(session_data)} sessions)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Per-Session UM MSE Trajectory

How does fit quality vary across sessions? A good model should have
consistently low MSE.

In [ ]:
if session_data:
    sess_idx = [d['session_idx'] for d in session_data]
    be_mses = [d['be_um_mse'] for d in session_data]
    sc_mses = [d['sc_um_mse'] for d in session_data]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(sess_idx, be_mses, 'o-', color='#4C72B0', label='BE', markersize=5)
    ax.plot(sess_idx, sc_mses, 'o-', color='#DD8452', label='SC', markersize=5)

    be_mean = np.nanmean(be_mses)
    sc_mean = np.nanmean(sc_mses)
    ax.axhline(be_mean, color='#4C72B0', ls='--', alpha=0.4)
    ax.axhline(sc_mean, color='#DD8452', ls='--', alpha=0.4)

    winner = 'BE' if be_mean < sc_mean else 'SC'
    ax.set_xlabel('Session index', fontsize=11)
    ax.set_ylabel('UM MSE', fontsize=11)
    ax.set_title(
        f'{EXAMPLE_ANIMAL} — Per-Session UM Error '
        f'(BE={be_mean:.5f}, SC={sc_mean:.5f} → {winner})',
        fontsize=12, fontweight='bold',
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()